# Task 1: Predictive Modeling

## Objective

Build a **regression** pipeline that predicts `Aggregate rating` from restaurant features.

This notebook answers the Cognifyz Level 3 predictive-modeling brief:

1. Define the target and confirm the problem type.
2. Inspect the engineered dataset and select features **without target leakage**.
3. Fit a reproducible preprocessing + train/test pipeline.
4. Experiment with Linear Regression, Decision Tree, and Random Forest.
5. Compare models with MAE, MSE, RMSE, and R² on the **test** set.
6. Inspect prediction diagnostics and interpret results, including limitations.

The original `Dataset.csv` is not modified. Modeling uses the Level 2 Task 3 engineered file.


## Dataset / Input

Primary input:

- `../data/engineered_dataset.csv` — copy of Level 2 Task 3 output (`9551` rows, `26` columns)

Target:

- `Aggregate rating` (continuous, range 0.0–4.9) → **regression**, not classification


## Imports


In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.dpi"] = 120

BASE_DIR = Path.cwd()
DATA_PATH = (BASE_DIR / "../data/engineered_dataset.csv").resolve()
FIG_DIR = BASE_DIR / "outputs" / "figures"
TAB_DIR = BASE_DIR / "outputs" / "tables"
MODEL_DIR = BASE_DIR / "models"
for d in (FIG_DIR, TAB_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Working directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Train/test split: {:.0%} / {:.0%}".format(1 - TEST_SIZE, TEST_SIZE))
print("random_state:", RANDOM_STATE)


Working directory: C:\Users\adity\Documents\Cognifyz\Level-3\Task-1_Predictive_Modeling
Dataset path: C:\Users\adity\Documents\Cognifyz\Level-3\data\engineered_dataset.csv
Train/test split: 80% / 20%
random_state: 42


## Load Dataset


In [2]:
df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:")
print(list(df.columns))
df.head()


Rows: 9551
Columns: 26
Column names:
['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes', 'Restaurant_Name_Length', 'Address_Length', 'Has_Table_Booking', 'Has_Online_Delivery', 'Cuisine_Count']


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Price range,Aggregate rating,Rating color,Rating text,Votes,Restaurant_Name_Length,Address_Length,Has_Table_Booking,Has_Online_Delivery,Cuisine_Count
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,3,4.8,Dark Green,Excellent,314,16,71,1,0,3
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,3,4.5,Dark Green,Excellent,591,16,67,1,0,1
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,4,4.4,Green,Very Good,270,22,56,1,0,4
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,4,4.9,Dark Green,Excellent,365,4,70,0,0,2
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,4,4.8,Dark Green,Excellent,229,11,64,1,0,2


## Data Validation

Confirm the target exists, inspect missing values, and document columns that must not be used as predictors.


In [3]:
print("Dtypes:")
print(df.dtypes.to_string())
print()
print("Missing values:")
print(df.isna().sum().to_string())
print()
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate Restaurant IDs:", int(df["Restaurant ID"].duplicated().sum()))
print()
print("Target summary (Aggregate rating):")
print(df["Aggregate rating"].describe().to_string())
print()
print("Target value counts (head):")
print(df["Aggregate rating"].value_counts().head(10).to_string())
print()
print("Zero ratings (Not rated band):", int((df["Aggregate rating"] == 0).sum()))


Dtypes:
Restaurant ID               int64
Restaurant Name               str
Country Code                int64
City                          str
Address                       str
Locality                      str
Locality Verbose              str
Longitude                 float64
Latitude                  float64
Cuisines                      str
Average Cost for two        int64
Currency                      str
Has Table booking             str
Has Online delivery           str
Is delivering now             str
Switch to order menu          str
Price range                 int64
Aggregate rating          float64
Rating color                  str
Rating text                   str
Votes                       int64
Restaurant_Name_Length      int64
Address_Length              int64
Has_Table_Booking           int64
Has_Online_Delivery         int64
Cuisine_Count               int64

Missing values:
Restaurant ID             0
Restaurant Name           0
Country Code              0
City   

## Leakage Check

`Rating color` and `Rating text` are labels of the same rating scale as `Aggregate rating` (White/Not rated = 0.0, Orange/Average = 2.5–3.4, etc.). Using them as features would leak the target.

Identifiers, free-text fields, constant columns, and the raw Yes/No service strings (already encoded in Level 2) are also excluded.


In [4]:
print("Rating text x Rating color vs Aggregate rating:")
rating_map = (
    df.groupby(["Rating text", "Rating color"], as_index=False)["Aggregate rating"]
    .agg(min_rating="min", max_rating="max", mean_rating="mean", n="count")
    .sort_values("min_rating")
)
display(rating_map)
rating_map.to_csv(TAB_DIR / "rating_band_mapping.csv", index=False)

print()
print("Switch to order menu (constant?):")
print(df["Switch to order menu"].value_counts().to_string())
print()
print("Is delivering now:")
print(df["Is delivering now"].value_counts().to_string())

excluded_leakage = ["Aggregate rating", "Rating color", "Rating text"]
excluded_id_text = [
    "Restaurant ID",
    "Restaurant Name",
    "Address",
    "Locality",
    "Locality Verbose",
    "Cuisines",
]
excluded_redundant = [
    "Has Table booking",
    "Has Online delivery",
    "Switch to order menu",
]
# City has 141 levels; one-hot would explode sparsity relative to 9.5k rows.
excluded_high_card = ["City"]

excluded = excluded_leakage + excluded_id_text + excluded_redundant + excluded_high_card
print()
print("Explicitly excluded columns:")
for c in excluded:
    print("-", c)


Rating text x Rating color vs Aggregate rating:


,Rating text,Rating color,min_rating,max_rating,mean_rating,n
3,Not rated,White,0.0,0.0,0.000000,2148
4,Poor,Red,1.8,2.4,2.297849,186
0,Average,Orange,2.5,3.4,3.051619,3737
2,Good,Yellow,3.5,3.9,3.683429,2100
5,Very Good,Green,4.0,4.4,4.168119,1079
1,Excellent,Dark Green,4.5,4.9,4.659801,301



Switch to order menu (constant?):
Switch to order menu
No    9551

Is delivering now:
Is delivering now
No     9517
Yes      34

Explicitly excluded columns:
- Aggregate rating
- Rating color
- Rating text
- Restaurant ID
- Restaurant Name
- Address
- Locality
- Locality Verbose
- Cuisines
- Has Table booking
- Has Online delivery
- Switch to order menu
- City


## Feature Selection

Predictors are restaurant characteristics that can exist independently of the posted aggregate rating.

**Numeric**

- `Price range` — ordinal cost band (strongest numeric association in Level 2)
- `Votes` — review volume (not a recoding of the rating itself; unrated rows have 0–3 votes)
- `Average Cost for two` — raw cost (currency-mixed; included with that caveat)
- `Cuisine_Count`, `Restaurant_Name_Length`, `Address_Length` — Level 2 engineered features
- `Longitude`, `Latitude` — location

**Binary / categorical**

- `Has_Table_Booking`, `Has_Online_Delivery` — Level 2 encodings
- `Is delivering now` — encoded in the pipeline
- `Country Code`, `Currency` — low-cardinality categoricals

`Votes` is **not** treated as leakage: it is a count of user votes, not a transform of `Aggregate rating`. It is still a post-opening popularity signal, so a production “cold start” model might drop it. That limitation is recorded at the end.


In [5]:
work = df.copy()
work["Is_Delivering_Now"] = (work["Is delivering now"] == "Yes").astype(int)

numeric_features = [
    "Price range",
    "Votes",
    "Average Cost for two",
    "Cuisine_Count",
    "Restaurant_Name_Length",
    "Address_Length",
    "Longitude",
    "Latitude",
    "Has_Table_Booking",
    "Has_Online_Delivery",
    "Is_Delivering_Now",
]
categorical_features = ["Country Code", "Currency"]
feature_cols = numeric_features + categorical_features
target_col = "Aggregate rating"

missing_cols = [c for c in feature_cols + [target_col] if c not in work.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

X = work[feature_cols]
y = work[target_col]

print("Feature matrix shape:", X.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Any missing in selected features?", int(X.isna().sum().sum()))
print()
print("Pearson correlation with target (numeric only):")
corr = work[numeric_features + [target_col]].corr(numeric_only=True)[target_col].sort_values(ascending=False)
print(corr.to_string())
corr.to_csv(TAB_DIR / "numeric_correlation_with_target.csv", header=["corr_with_aggregate_rating"])


Feature matrix shape: (9551, 13)
Numeric features: ['Price range', 'Votes', 'Average Cost for two', 'Cuisine_Count', 'Restaurant_Name_Length', 'Address_Length', 'Longitude', 'Latitude', 'Has_Table_Booking', 'Has_Online_Delivery', 'Is_Delivering_Now']
Categorical features: ['Country Code', 'Currency']
Any missing in selected features? 0

Pearson correlation with target (numeric only):
Aggregate rating          1.000000
Price range               0.437944
Votes                     0.313691
Cuisine_Count             0.248404
Has_Online_Delivery       0.225699
Has_Table_Booking         0.189998
Average Cost for two      0.051792
Is_Delivering_Now         0.019180
Address_Length            0.002334
Latitude                  0.000516
Restaurant_Name_Length   -0.035178
Longitude                -0.116818


## Preparation — Train/Test Split and Preprocessing Pipeline

- Split: **80% train / 20% test**, `random_state=42`
- Numeric columns: `StandardScaler` fitted on **train only**
- Categorical columns: `OneHotEncoder(handle_unknown="ignore")` fitted on **train only**
- No statistics are computed from the test set


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Train rows:", len(X_train), f"({len(X_train) / len(X):.1%})")
print("Test rows:", len(X_test), f"({len(X_test) / len(X):.1%})")
print("Train target mean:", round(float(y_train.mean()), 4), "std:", round(float(y_train.std()), 4))
print("Test target mean:", round(float(y_test.mean()), 4), "std:", round(float(y_test.std()), 4))

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="drop",
)

split_info = pd.DataFrame(
    {
        "set": ["train", "test"],
        "n_rows": [len(X_train), len(X_test)],
        "share": [len(X_train) / len(X), len(X_test) / len(X)],
        "target_mean": [float(y_train.mean()), float(y_test.mean())],
        "target_std": [float(y_train.std()), float(y_test.std())],
    }
)
split_info.to_csv(TAB_DIR / "train_test_split.csv", index=False)
split_info


Train rows: 7640 (80.0%)
Test rows: 1911 (20.0%)
Train target mean: 2.66 std: 1.5182


Test target mean: 2.6919 std: 1.5091


,set,n_rows,share,target_mean,target_std
0,train,7640,0.799916,2.659987,1.518231
1,test,1911,0.200084,2.691889,1.509073


## Models

| Model | Role |
|---|---|
| Linear Regression | Linear baseline |
| Decision Tree Regressor | Nonlinear, single tree (prone to overfit) |
| Random Forest Regressor | Ensemble of trees |

Hyperparameters are kept modest. This is a comparison of algorithm families, not a large benchmark.


In [7]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

fitted = {}
predictions = {}
metric_rows = []

for name, estimator in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", estimator),
        ]
    )
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    fitted[name] = pipe
    predictions[name] = y_pred

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_test, y_pred)
    metric_rows.append(
        {
            "Model": name,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2,
        }
    )
    joblib.dump(pipe, MODEL_DIR / f"{name.lower().replace(' ', '_')}.joblib")
    print(f"{name:20s}  MAE={mae:.4f}  MSE={mse:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TAB_DIR / "model_comparison.csv", index=False)
metrics


Linear Regression     MAE=1.0333  MSE=1.5520  RMSE=1.2458  R2=0.3181


Decision Tree         MAE=0.2847  MSE=0.1904  RMSE=0.4363  R2=0.9164


Random Forest         MAE=0.1968  MSE=0.0910  RMSE=0.3016  R2=0.9600


,Model,MAE,MSE,RMSE,R2
0,Linear Regression,1.033287,1.552047,1.245812,0.318114
1,Decision Tree,0.284720,0.190387,0.436334,0.916354
2,Random Forest,0.196822,0.090970,0.301612,0.960033


## Evaluation Metrics

Computed on the **held-out test set only**.

- **MAE** — mean absolute error in rating points. Easy to interpret (same units as the target).
- **MSE** — mean squared error; penalizes large misses more than MAE.
- **RMSE** — square root of MSE; also in rating points, more sensitive to outliers than MAE.
- **R²** — fraction of test-set variance explained relative to predicting the mean. `1` is perfect; `0` matches a constant-mean baseline; negative is worse than that baseline.

“Best” depends on the metric. Lower MAE/RMSE is better; higher R² is better. We will not declare a winner from a single number without context.


In [8]:
metrics_display = metrics.copy()
for c in ["MAE", "MSE", "RMSE", "R2"]:
    metrics_display[c] = metrics_display[c].map(lambda x: round(float(x), 4))

print("Test-set comparison:")
display(metrics_display)

best_rmse_name = metrics.loc[metrics["RMSE"].idxmin(), "Model"]
best_r2_name = metrics.loc[metrics["R2"].idxmax(), "Model"]
print(f"Lowest RMSE: {best_rmse_name}")
print(f"Highest R2:   {best_r2_name}")


Test-set comparison:


,Model,MAE,MSE,RMSE,R2
0,Linear Regression,1.0333,1.5520,1.2458,0.3181
1,Decision Tree,0.2847,0.1904,0.4363,0.9164
2,Random Forest,0.1968,0.0910,0.3016,0.9600


Lowest RMSE: Random Forest
Highest R2:   Random Forest


## Diagnostic Visualizations


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
lims = [float(y_test.min()), float(y_test.max())]
for ax, name in zip(axes, models.keys()):
    y_pred = predictions[name]
    ax.scatter(y_test, y_pred, alpha=0.35, s=12, edgecolors="none")
    ax.plot(lims, lims, color="black", linewidth=1, linestyle="--", label="Perfect prediction")
    ax.set_title(name)
    ax.set_xlabel("Actual Aggregate rating")
    ax.set_ylabel("Predicted Aggregate rating")
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 5)
    ax.legend(loc="upper left", fontsize=8)

fig.suptitle("Actual vs Predicted Aggregate Rating (test set)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_20660\899279072.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, name in zip(axes, models.keys()):
    resid = y_test.to_numpy() - predictions[name]
    ax.scatter(predictions[name], resid, alpha=0.35, s=12, edgecolors="none")
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Predicted Aggregate rating")
    ax.set_ylabel("Residual (actual − predicted)")

fig.suptitle("Residual Analysis (test set)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "residuals.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_20660\3065652418.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, name in zip(axes, models.keys()):
    resid = y_test.to_numpy() - predictions[name]
    ax.hist(resid, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(name)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")

fig.suptitle("Residual Distribution (test set)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "residual_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_20660\4109333745.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Random Forest Feature Importance

Importance is a **split-based association** inside this forest, not a causal ranking. Correlated features can share or steal importance.


In [12]:
rf_pipe = fitted["Random Forest"]
ohe = rf_pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(categorical_features))
feature_names = numeric_features + cat_names
importances = rf_pipe.named_steps["model"].feature_importances_
imp = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
imp.to_csv(TAB_DIR / "random_forest_feature_importance.csv", index=False)
display(imp.head(15))

top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["feature"], top["importance"], color="steelblue")
ax.set_title("Random Forest Feature Importance (top 15)")
ax.set_xlabel("Mean impurity decrease (relative importance)")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(FIG_DIR / "rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


,feature,importance
0,Votes,0.946639
1,Longitude,0.015338
2,Latitude,0.011585
3,Address_Length,0.006658
4,Average Cost for two,0.006113
5,Restaurant_Name_Length,0.005259
6,Cuisine_Count,0.002276
7,Currency_Indian Rupees(Rs.),0.001439
8,Country Code_1,0.001235
9,Has_Online_Delivery,0.000985


C:\Users\adity\AppData\Local\Temp\ipykernel_20660\3187911554.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sensitivity — Models Without `Votes`

Random Forest importance is dominated by `Votes`. That column is not a recoding of `Aggregate rating`, but unrated restaurants almost all have 0–3 votes, so `Votes` is a strong proxy for “has a rating.”

The same three algorithms are retrained on the identical split with `Votes` removed. This is a sensitivity check, not a fourth algorithm family.


In [13]:
numeric_no_votes = [c for c in numeric_features if c != "Votes"]
preprocess_no_votes = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_no_votes),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="drop",
)

X_train_nv = X_train.drop(columns=["Votes"])
X_test_nv = X_test.drop(columns=["Votes"])

ablation_rows = []
for name, estimator in models.items():
    # Fresh estimator instances; the dict holds already-fitted objects
    if name == "Linear Regression":
        est = LinearRegression()
    elif name == "Decision Tree":
        est = DecisionTreeRegressor(random_state=RANDOM_STATE)
    else:
        est = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    pipe = Pipeline([("preprocess", preprocess_no_votes), ("model", est)])
    pipe.fit(X_train_nv, y_train)
    y_pred = pipe.predict(X_test_nv)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_test, y_pred)
    ablation_rows.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2})
    print(f"{name:20s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(TAB_DIR / "model_comparison_without_votes.csv", index=False)

compare = metrics.merge(ablation, on="Model", suffixes=("_with_votes", "_without_votes"))
compare.to_csv(TAB_DIR / "model_comparison_votes_ablation.csv", index=False)
display(compare)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width / 2, metrics["R2"], width, label="With Votes", color="steelblue")
ax.bar(x + width / 2, ablation["R2"], width, label="Without Votes", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(metrics["Model"], rotation=15)
ax.set_ylabel("Test R²")
ax.set_title("Sensitivity: Test R² With vs Without Votes")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "r2_votes_ablation.png", dpi=150, bbox_inches="tight")
plt.show()


Linear Regression     MAE=1.0409  RMSE=1.2573  R2=0.3054
Decision Tree         MAE=0.9643  RMSE=1.5445  R2=-0.0481


Random Forest         MAE=0.8227  RMSE=1.1057  R2=0.4628


,Model,MAE_with_votes,MSE_with_votes,RMSE_with_votes,R2_with_votes,MAE_without_votes,MSE_without_votes,RMSE_without_votes,R2_without_votes
0,Linear Regression,1.033287,1.552047,1.245812,0.318114,1.040921,1.580896,1.257337,0.305439
1,Decision Tree,0.284720,0.190387,0.436334,0.916354,0.964329,2.385499,1.544506,-0.048060
2,Random Forest,0.196822,0.090970,0.301612,0.960033,0.822658,1.222627,1.105724,0.462844


C:\Users\adity\AppData\Local\Temp\ipykernel_20660\1956525273.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results


In [14]:
print("Saved comparison table:")
print((TAB_DIR / "model_comparison.csv").resolve())
display(metrics_display)

pred_frame = pd.DataFrame({"actual": y_test.to_numpy()})
for name, y_pred in predictions.items():
    pred_frame[name] = y_pred
pred_frame.to_csv(TAB_DIR / "test_predictions.csv", index=False)
print("Saved test predictions:", TAB_DIR / "test_predictions.csv")
print("Saved models:", list(MODEL_DIR.glob("*.joblib")))


Saved comparison table:
C:\Users\adity\Documents\Cognifyz\Level-3\Task-1_Predictive_Modeling\outputs\tables\model_comparison.csv


,Model,MAE,MSE,RMSE,R2
0,Linear Regression,1.0333,1.5520,1.2458,0.3181
1,Decision Tree,0.2847,0.1904,0.4363,0.9164
2,Random Forest,0.1968,0.0910,0.3016,0.9600


Saved test predictions: C:\Users\adity\Documents\Cognifyz\Level-3\Task-1_Predictive_Modeling\outputs\tables\test_predictions.csv
Saved models: [WindowsPath('C:/Users/adity/Documents/Cognifyz/Level-3/Task-1_Predictive_Modeling/models/decision_tree.joblib'), WindowsPath('C:/Users/adity/Documents/Cognifyz/Level-3/Task-1_Predictive_Modeling/models/linear_regression.joblib'), WindowsPath('C:/Users/adity/Documents/Cognifyz/Level-3/Task-1_Predictive_Modeling/models/random_forest.joblib')]


## Findings

1. This is a **regression** problem. The target is `Aggregate rating`, a continuous score.
2. `Rating color` and `Rating text` map onto disjoint rating bands, so they were excluded as leakage.
3. Three algorithm families were trained on the same 80/20 split and the same preprocessing pipeline.
4. Metrics above are computed from **actual test predictions**, not placeholders.
5. Residual plots show whether errors are symmetric and whether the unrated (`0.0`) mass is hard to predict.
6. Random Forest importance is dominated by `Votes`. The no-Votes ablation shows how much of the headline R² is a “has been rated” proxy rather than menu/price/location signal.
7. Feature importance is not causal.


## Limitations

- The dataset is geographically skewed (most rows are India / New Delhi). Generalization to other markets is not demonstrated.
- `Average Cost for two` mixes currencies. `Price range` is the more comparable cost signal.
- About 2,148 restaurants are unrated (`Aggregate rating = 0`). That spike is a data convention, not a true 0-star experience, and it inflates error if the model cannot separate “not rated” from low-rated.
- `Votes` is observed after customers interact with the restaurant. A brand-new restaurant would not have this signal.
- No hyperparameter search or nested cross-validation was run. Rankings could shift with tuning.
- Feature importance is not causal.
- The model predicts a rating score, **not** restaurant success, revenue, or quality of food.


## Conclusion

A leakage-checked regression pipeline was trained to predict `Aggregate rating` from the Level 2 engineered dataset. Linear Regression, a Decision Tree, and a Random Forest were evaluated on a fixed 20% test split using MAE, MSE, RMSE, and R², with actual-vs-predicted and residual diagnostics. Results should be read as in-sample predictive comparisons on this table, not as a claim that ratings (or business success) can be forecast in deployment without the limitations above.
